# 📱 WhatsApp Outreach Tool — L&D Designs
**Turns your leads spreadsheet into a click-to-send WhatsApp dashboard.**

Each business gets a button. Click it → WhatsApp opens with your message already typed → just hit Send.

---
**How to use:**
1. Run Cell 1 — upload your `wigan_hair_leads_*.xlsx` file
2. Run Cell 2 — generates your outreach dashboard
3. Run Cell 3 — downloads the dashboard as an HTML file
4. Open the downloaded HTML file in your browser and start messaging

In [ ]:
# ── Cell 1: Upload your leads spreadsheet ────────────────────────────────
from google.colab import files
import openpyxl

print('Select your wigan_hair_leads_*.xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb['Leads']

headers = [cell.value for cell in ws[1]]
leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0]:  # skip empty rows
        leads.append(dict(zip(headers, row)))

# Only keep leads that have a phone number
leads_with_phone = [l for l in leads if l.get('Phone Number')]

print(f'\n✅ Loaded {len(leads)} leads total')
print(f'   {len(leads_with_phone)} have a phone number (these get WhatsApp buttons)')
print(f'   {len(leads) - len(leads_with_phone)} have no phone number (shown as info only)')

In [ ]:
# ── Cell 2: Generate the outreach dashboard ───────────────────────────────
import re
from urllib.parse import quote

# ── YOUR MESSAGE TEMPLATE ─────────────────────────────────────────────────
# {name} is replaced with the business name automatically
MESSAGE_TEMPLATE = """Hi! 👋 I came across {name} and noticed you don't have a website yet.

I'm Dylan from L&D Designs — I build professional websites for local barbers and hairdressers across the Wigan area.

A website means new customers can find you on Google, see your work, and get in touch easily. Most of my sites are live within 1–2 weeks 🚀

Would you be up for a free quote? No pressure at all 😊

— Dylan, L&D Designs"""

OUTDATED_MESSAGE_TEMPLATE = """Hi! 👋 I came across {name} and noticed your website could do with a refresh.

I'm Dylan from L&D Designs — I build modern websites for local barbers and hairdressers across the Wigan area.

An updated site helps you rank higher on Google and gives customers a much better first impression 💈

Happy to offer you a free quote with no obligation at all 😊

— Dylan, L&D Designs"""
# ─────────────────────────────────────────────────────────────────────────

def clean_phone(phone):
    """Convert UK phone number to international WhatsApp format (e.g. 447911123456)"""
    if not phone:
        return ''
    digits = re.sub(r'[^\d+]', '', str(phone))
    digits = digits.lstrip('+')
    if digits.startswith('0'):
        digits = '44' + digits[1:]
    elif not digits.startswith('44'):
        digits = '44' + digits
    return digits

def make_wa_link(phone, message):
    clean = clean_phone(phone)
    if not clean:
        return ''
    return f'https://wa.me/{clean}?text={quote(message)}'

STATUS_COLOURS = {
    'NONE':        ('#ffd6d6', '#c0392b', '🔴 No website'),
    'SOCIAL ONLY': ('#d6eaff', '#2980b9', '🔵 Social only'),
    'OUTDATED':    ('#fff2cc', '#e67e22', '🟡 Outdated site'),
    'ERROR':       ('#e8e8e8', '#7f8c8d', '⚫ Site error'),
}

def make_card(lead):
    name    = lead.get('Business Name', 'Unknown')
    phone   = str(lead.get('Phone Number', '') or '')
    email   = lead.get('Email Address', '') or ''
    status  = str(lead.get('Website Status', '') or '').upper().strip()
    address = lead.get('Address', '') or ''
    dist    = lead.get('Distance (miles)', '') or ''
    website = lead.get('Website / Social URL', '') or ''
    notes   = lead.get('Notes', '') or ''

    bg, accent, badge = STATUS_COLOURS.get(status, ('#f7f7f7', '#555', status))

    # Pick message based on status
    if 'OUTDATED' in status or 'SOCIAL' in status:
        message = OUTDATED_MESSAGE_TEMPLATE.format(name=name)
    else:
        message = MESSAGE_TEMPLATE.format(name=name)

    wa_link = make_wa_link(phone, message)

    wa_button = ''
    if wa_link:
        wa_button = f'<a class="wa-btn" href="{wa_link}" target="_blank">📱 Send WhatsApp</a>'
    else:
        wa_button = '<span class="no-phone">No phone number</span>'

    email_line = f'<div class="detail">✉️ {email}</div>' if email else ''
    website_line = f'<div class="detail"><a href="{website}" target="_blank">🌐 {website[:45]}{'...' if len(website)>45 else ''}</a></div>' if website else ''
    notes_line = f'<div class="detail muted">{notes}</div>' if notes else ''

    return f"""
    <div class="card" style="border-left:4px solid {accent}; background:{bg}">
      <div class="card-header">
        <div>
          <div class="biz-name">{name}</div>
          <div class="badge" style="color:{accent}">{badge}</div>
        </div>
        <div class="dist">{dist} mi</div>
      </div>
      <div class="detail">📍 {address}</div>
      <div class="detail">📞 {phone if phone else '—'}</div>
      {email_line}
      {website_line}
      {notes_line}
      <div class="actions">
        {wa_button}
      </div>
    </div>"""

# Build stats
total = len(leads)
with_phone = len(leads_with_phone)
counts = {}
for l in leads:
    s = str(l.get('Website Status','')).upper().strip()
    counts[s] = counts.get(s,0) + 1

cards_html = '\n'.join(make_card(l) for l in leads)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>L&D Designs — WhatsApp Outreach</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
         background: #f0f2f5; color: #333; }}
  .header {{ background: #1a2035; color: white; padding: 20px 24px; }}
  .header h1 {{ font-size: 1.4rem; }}
  .header p  {{ opacity: 0.7; font-size: 0.9rem; margin-top: 4px; }}
  .stats {{ display: flex; gap: 12px; flex-wrap: wrap; padding: 16px 24px;
             background: white; border-bottom: 1px solid #ddd; }}
  .stat {{ background: #f7f7f7; border-radius: 8px; padding: 10px 16px; text-align: center; }}
  .stat .n {{ font-size: 1.6rem; font-weight: bold; color: #1a2035; }}
  .stat .l {{ font-size: 0.75rem; color: #888; }}
  .filter-bar {{ padding: 12px 24px; background: white;
                  border-bottom: 1px solid #ddd; display: flex; gap: 8px; flex-wrap: wrap; }}
  .filter-btn {{ padding: 6px 14px; border: 2px solid #ddd; border-radius: 20px;
                  background: white; cursor: pointer; font-size: 0.82rem; transition: all .2s; }}
  .filter-btn.active {{ border-color: #1a2035; background: #1a2035; color: white; }}
  .grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(340px, 1fr));
            gap: 14px; padding: 20px 24px; }}
  .card {{ background: white; border-radius: 10px; padding: 16px;
            box-shadow: 0 1px 4px rgba(0,0,0,.08); }}
  .card-header {{ display: flex; justify-content: space-between; align-items: flex-start;
                   margin-bottom: 10px; }}
  .biz-name {{ font-weight: 700; font-size: 1rem; line-height: 1.3; }}
  .badge {{ font-size: 0.75rem; font-weight: 600; margin-top: 3px; }}
  .dist {{ font-size: 0.8rem; color: #888; white-space: nowrap; padding-left: 8px; }}
  .detail {{ font-size: 0.82rem; color: #555; margin: 3px 0; word-break: break-word; }}
  .detail a {{ color: #2980b9; text-decoration: none; }}
  .muted {{ color: #999; font-style: italic; }}
  .actions {{ margin-top: 12px; }}
  .wa-btn {{ display: inline-block; background: #25D366; color: white;
              padding: 8px 18px; border-radius: 6px; text-decoration: none;
              font-size: 0.88rem; font-weight: 600; transition: background .2s; }}
  .wa-btn:hover {{ background: #1ebe5d; }}
  .no-phone {{ font-size: 0.82rem; color: #bbb; }}
  .sent {{ opacity: 0.45; }}
  .mark-btn {{ margin-left: 10px; background: none; border: 1px solid #ccc;
                color: #888; padding: 7px 12px; border-radius: 6px; cursor: pointer;
                font-size: 0.82rem; }}
  .progress {{ padding: 8px 24px; background: #fff; border-bottom: 1px solid #eee;
                font-size: 0.85rem; color: #555; }}
</style>
</head>
<body>

<div class="header">
  <h1>💈 L&D Designs — WhatsApp Outreach</h1>
  <p>50-mile radius of Wigan &nbsp;·&nbsp; Generated {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
</div>

<div class="stats">
  <div class="stat"><div class="n">{total}</div><div class="l">Total Leads</div></div>
  <div class="stat"><div class="n">{with_phone}</div><div class="l">Have Phone</div></div>
  <div class="stat"><div class="n" style="color:#c0392b">{counts.get('NONE',0)}</div><div class="l">No Website</div></div>
  <div class="stat"><div class="n" style="color:#2980b9">{counts.get('SOCIAL ONLY',0)}</div><div class="l">Social Only</div></div>
  <div class="stat"><div class="n" style="color:#e67e22">{counts.get('OUTDATED',0)}</div><div class="l">Outdated Site</div></div>
</div>

<div class="progress" id="progress">Sent: <strong id="sent-count">0</strong> / {with_phone}</div>

<div class="filter-bar">
  <button class="filter-btn active" onclick="filter('all')">All</button>
  <button class="filter-btn" onclick="filter('none')">🔴 No Website</button>
  <button class="filter-btn" onclick="filter('social')">🔵 Social Only</button>
  <button class="filter-btn" onclick="filter('outdated')">🟡 Outdated</button>
  <button class="filter-btn" onclick="filter('unsent')">📤 Not Sent Yet</button>
</div>

<div class="grid" id="grid">
{cards_html}
</div>

<script>
  // Mark as sent when WhatsApp button is clicked
  document.querySelectorAll('.wa-btn').forEach(btn => {{
    btn.addEventListener('click', function() {{
      const card = this.closest('.card');
      setTimeout(() => {{
        card.classList.add('sent');
        // Add undo button
        if (!card.querySelector('.mark-btn')) {{
          const undo = document.createElement('button');
          undo.className = 'mark-btn';
          undo.textContent = '↩ Undo';
          undo.onclick = () => {{ card.classList.remove('sent'); undo.remove(); updateCount(); }};
          this.parentNode.appendChild(undo);
        }}
        updateCount();
      }}, 1500);
    }});
  }});

  function updateCount() {{
    document.getElementById('sent-count').textContent = document.querySelectorAll('.card.sent').length;
  }}

  function filter(type) {{
    document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
    event.target.classList.add('active');
    document.querySelectorAll('.card').forEach(card => {{
      const badge = (card.querySelector('.badge')?.textContent || '').toLowerCase();
      const sent  = card.classList.contains('sent');
      let show = true;
      if (type === 'none')     show = badge.includes('no website');
      if (type === 'social')   show = badge.includes('social');
      if (type === 'outdated') show = badge.includes('outdated');
      if (type === 'unsent')   show = !sent && card.querySelector('.wa-btn');
      card.style.display = show ? '' : 'none';
    }});
  }}
</script>
</body>
</html>"""

with open('outreach_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'✅ Dashboard generated with {len(leads)} leads')
print(f'   {with_phone} businesses have WhatsApp buttons')

In [ ]:
# ── Cell 3: Download the dashboard ───────────────────────────────────────
from google.colab import files
files.download('outreach_dashboard.html')
print('✅ Download started!')
print()
print('HOW TO USE:')
print('1. Open the downloaded outreach_dashboard.html file in your browser')
print('2. Click "📱 Send WhatsApp" on each business')
print('3. WhatsApp opens with your message already typed — just hit Send')
print('4. The card fades out so you know who you\'ve messaged')